# Face Feature Extractors — EmotiEffNet / FaRL

Two complementary frozen face encoders for affect, personality and sentiment
pipelines. Both are **frame-wise** — no 16-frame clip buffering, unlike MARLIN —
so they run comfortably in interactive settings and accept either a single face
image or a face `Track`.

| Model | Wrapper | Output | Input | Pre-training | Weights licence |
|---|---|---|---|---|---|
| EmotiEffNet B0/B2 | `EmotiEffNetWrapper` | 1280 / 1408-d | 224 / 260 | VGGFace2 → AffectNet (8 expressions) | Apache-2.0 |
| FaRL ViT-B/16 | `FarlWrapper` | 512-d | 224 | LAION-Face 20M (image–text + MIM) | MIT |

**How they differ.** EmotiEffNet is *supervised* on expression labels, so its
features are tuned for emotion/valence–arousal but discard cues its 8-class head
never needed. FaRL is *self-supervised* and vision-language pretrained, so it
keeps identity-ish, appearance and style cues — the things apparent-personality
prediction leans on — and its paper reports strong transfer in the low-data
regimes that personality corpora live in. They are meant to be **concatenated**,
not chosen between.

This notebook downloads **real pretrained weights** (the test suite deliberately
does not) and checks the features actually behave — that is the point of running
it.


## Setup

In [ ]:
from pathlib import Path

import torch
import torch.nn.functional as F

from exordium.video.deep import EmotiEffNetWrapper, FarlWrapper

DEVICE_ID = -1  # -1 → CPU; set to a GPU index to use CUDA

FIXTURES = Path("../tests/fixtures/image")
FACE = FIXTURES / "face.jpg"
EMMA = FIXTURES / "emma.jpg"
CAT = FIXTURES / "cat_tie.jpg"  # not a face → the negative control

print("face:", FACE.exists(), "| emma:", EMMA.exists(), "| cat:", CAT.exists())

## 1. Single face image

Every wrapper takes a path, a numpy array, or a tensor, and returns `(B, D)`.

In [ ]:
emotieffnet = EmotiEffNetWrapper(model_name="enet_b0_8_best_vgaf", device_id=DEVICE_ID)
farl = FarlWrapper(model_name="ep64", device_id=DEVICE_ID)

models = {"EmotiEffNet-B0": emotieffnet, "FaRL-B/16": farl}

for name, model in models.items():
    feat = model(FACE)
    print(f"{name:15s} {tuple(feat.shape)}  dim={model.feature_dim}")

### Does it actually work?

Loading without error proves nothing — a randomly-initialised model produces
correctly-shaped features too. So how do you *check* a frozen feature extractor?

The property that separates trained from untrained features is whether a
**semantic label is readable** from them. The check below does that against
ground truth for EmotiEffNet; the section after it is candid about why the same
trick does not work for the self-supervised pair, and what to do instead.

#### Check 1 — EmotiEffNet against ground truth

EmotiEffNet is supervised, so it carries its own semantic label set: the 8
AffectNet expression classes. Re-attach its classifier head and the features are
verifiable directly against something we can look at with our own eyes — a thing
random weights cannot fake.

In [ ]:
import timm
import torchvision.transforms.functional as TF

from exordium import WEIGHT_DIR
from exordium.video.core.io import to_uint8_tensor
from exordium.video.deep.emotieffnet import _MODELS, _load_state_dict_from_pickle

AFFECTNET = ["Anger", "Contempt", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]

# Same checkpoint the wrapper loads, but keeping the classification head that
# EmotiEffNetWrapper replaces with Identity.
variant = "enet_b0_8_best_vgaf"
cfg = _MODELS[variant]
classifier = timm.create_model(cfg["timm_name"], pretrained=False, num_classes=8)
classifier.load_state_dict(
    _load_state_dict_from_pickle(str(WEIGHT_DIR / "emotieffnet" / f"{variant}.pt"))
)
classifier.eval()

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

for path in (FACE, EMMA):
    x = TF.resize(to_uint8_tensor(path), [224, 224], antialias=True).float().div(255)
    with torch.inference_mode():
        probs = classifier((x - IMAGENET_MEAN) / IMAGENET_STD).softmax(-1)[0]
    top3 = probs.argsort(descending=True)[:3]
    print(f"{path.name:10s} " + "   ".join(f"{AFFECTNET[i]}={probs[i]:.2f}" for i in top3))

Open `tests/fixtures/image/face.jpg` and compare: wide eyes, raised brows, pursed
lips. A confident *Surprise* is the right call. The checkpoint is loaded
correctly and the penultimate features the wrapper returns are the ones feeding
that decision.

#### Check 2 — determinism and the limits of a two-image sanity check

FaRL is self-supervised, so it has no ground-truth head to check against. What
*can* be verified cheaply is that it is deterministic, produces unit-norm
embeddings, and is sensitive to the input:


In [ ]:
for name, model in models.items():
    with torch.inference_mode():
        a = model(FACE)
        b = model(FACE)  # same input, twice
        c = model(EMMA)  # different face
    print(
        f"{name:15s} deterministic={torch.allclose(a, b)}  "
        f"‖e‖={float(F.normalize(a.float(), dim=-1).norm()):.3f}  "
        f"differs on a different face={not torch.allclose(a, c)}"
    )

**And now the honest caveat, because it matters more than a green checkmark.**

It is tempting to go further here — cluster the two faces against the cat, or
probe *face vs. scrambled-face* — and declare the weights validated. Those tests
were tried while writing this notebook and **every one of them is also passed by
a randomly-initialised model**, so they prove nothing. A random ViT is still a
smooth, high-dimensional projection: near-identical inputs give near-identical
outputs, distinct inputs stay linearly separable, and raw cosine geometry tracks
colour and texture statistics rather than anything learned. Untrained features
are not noise; they are merely *not organised around anything semantic*.

Separating a trained face encoder from an untrained one needs a label that only
face knowledge can predict, measured over **many identities under matched
conditions** — the two fixture images in this repo cannot do it, and a notebook
that claimed otherwise would be lying to you.

So: Check 1 is the real evidence that the checkpoint plumbing is correct
(a supervised head, verifiable against ground truth, on shared preprocessing).
For FaRL, the honest validation is the one you run on *your* data:

```python
feats = model.track_to_feature(track)["features"]   # (T, D), frozen
# fit a linear/ridge head on your labels; compare against the other encoders
```

That ablation — which encoder, or which concatenation, linearly predicts *your*
emotion / personality / valence target — is the only comparison that answers the
question you actually have, and it is far cheaper than fine-tuning to find out.

## 2. Face track

`track_to_feature` runs the encoder over every non-interpolated detection in a
`Track` and returns aligned `frame_ids` + `features`. Crops are taken per-frame,
so their sizes drift as the box moves — the wrappers resize internally, so a
ragged track just works.

In [ ]:
from exordium.video.core.detection import DetectionFromTorchTensor, Track


def demo_track(num_frames: int = 8) -> Track:
    """A synthetic track whose box drifts in size, as a tracked face does."""
    frame = torch.randint(0, 255, (3, 400, 400), dtype=torch.uint8)
    landmarks = torch.zeros(5, 2, dtype=torch.long)
    track = Track(track_id=0)
    for fid in range(num_frames):
        side = 100 + (fid % 5) * 12  # crop size changes frame to frame
        xy = 200 - side // 2
        track.add(
            DetectionFromTorchTensor(
                frame_id=fid,
                source=frame,
                score=0.99,
                bb_xywh=torch.tensor([xy, xy, side, side], dtype=torch.long),
                landmarks=landmarks,
            )
        )
    return track


track = demo_track()
print(f"track with {len(track)} detections, crop sizes vary per frame\n")

for name, model in models.items():
    out = model.track_to_feature(track, batch_size=4)
    print(
        f"{name:15s} features={tuple(out['features'].shape)}  frame_ids={out['frame_ids'].tolist()}"
    )

On a real video you would build the track with the face detector / protagonist
extractor (see `demo_video_protagonist.ipynb`) and pass `output_path=...` to
cache the features to a safetensors file.

## 3. Fusing the two

The point of these encoders is to be concatenated — with each other and with the
non-face streams (DINOv2 on the full frame, WavLM / emotion2vec+ on audio,
XLM-R on the transcript) — and fed to a temporal head.


In [ ]:
with torch.inference_mode():
    face_feature = torch.cat([model(FACE).float() for model in models.values()], dim=-1)

dims = " + ".join(str(m.feature_dim) for m in models.values())
print(f"fused face feature: {dims} = {face_feature.shape[-1]}-d")
print(f"shape: {tuple(face_feature.shape)}")

Per frame that is a 1792-d face descriptor (1280 + 512). Concatenate it with
your full-frame and audio streams, aggregate over time (mean/attention/TCN
/ BiLSTM), and train the head — which is where the temporal modelling belongs
once the backbones are frozen.
